# 1. Setup environment (CPU/GPU/TPU) and import packages

In [ ]:
## For TPU environment (install missing packages / reinstall tensorflow to solve NaN topic during training / restart kernel)

import IPython
import tensorflow as tf

if len(tf.config.experimental.list_logical_devices('TPU'))>0:
    !pip install -q tensorflow-tpu -f https://storage.googleapis.com/libtpu-tf-releases/index.html --force-reinstall
    !pip install -q pydot
    !pip install -q -U keras-tuner
    !pip install xgboost
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
## Import packages

# General packages
import time

# Main data handling and visualization packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import uniform

# Scikit-Learn preprocessing modules
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split

# Scikit-Learn ML + xgboost modules
from sklearn.metrics import mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import SGDRegressor
import xgboost as xgboost

# Tensorflow modules
import tensorflow as tf
import keras_tuner as kt

In [ ]:
## Detect hardware (CPU/GPU/TPU), setup environment and return appropriate distribution strategy

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu='local') # set tpu is local as it should be available in the VM
    print('✅ Running on TPU ', tpu.master())
except:
    print('❌ Using CPU/GPU')
    tpu = None

if tpu:
    strategy = tf.distribute.TPUStrategy(tpu)
else:
    strategy = tf.distribute.get_strategy() # default distribution strategy in Tensorflow. Works on CPU and single GPU.

print("REPLICAS: ", strategy.num_replicas_in_sync)

# 2. Load and preprocess data into datasets

In [ ]:
## Load data

# Read csv files
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv")
oil_price = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv")

# Preprocess holidays data
holidays = holidays[holidays['transferred'] == False] # Filter out transferred holidays (redundant)
holidays = holidays.replace(to_replace=['Transfer', 'Bridge', 'Additional'], value='Holiday') # Rename 'Transfer/Bridge/Additional' to 'Holiday'
holidays['Holiday_Gen'] = holidays['type']=='Holiday'
holidays['Event_Gen'] = holidays['type']=='Event'
holidays['Work_Day_Gen'] = holidays['type']=='Work Day'
holidays.drop(columns=["type", "transferred"], inplace=True)

# Preprocess oil price data
oil_price.loc[0, "dcoilwtico"] = oil_price.loc[1, "dcoilwtico"]
oil_price.ffill(inplace=True)
oil_price.bfill(inplace=True)

In [ ]:
## Preprocess data

# Initiating scaling instances
scalers = {'scaler_onpromotion' : MinMaxScaler(), 'scaler_day' : MinMaxScaler(), 'scaler_month' : MinMaxScaler(),
           'scaler_year' : MinMaxScaler(), 'scaler_dcoilwtico' : MinMaxScaler(), 'scaler_DoW' : MinMaxScaler()}

def process_data(data, stores, transactions, holidays, oil_price, fit_scaler, scalers):
    # Merge stores into data to substitute store ids with location data
    data = data.merge(stores, how='left', left_on='store_nbr', right_on='store_nbr')
    
    # Merge holidays into data to extend location names with location related holidays
    data = data.merge(holidays, how='left', left_on='date', right_on='date')
    data['Holiday'] = ((data['Holiday_Gen'] == True)
                       & ((data['city'] == data['locale_name'])
                          | (data['state'] == data['locale_name'])
                          | ('Ecuador' == data['locale_name']))).astype(dtype='float64')
    data['Event'] = ((data['Event_Gen'] == True)
                       & ((data['city'] == data['locale_name'])
                          | (data['state'] == data['locale_name'])
                          | ('Ecuador' == data['locale_name']))).astype(dtype='float64')
    data['Work_Day'] = ((data['Work_Day_Gen'] == True)
                        & ((data['city'] == data['locale_name'])
                           | (data['state'] == data['locale_name'])
                           | ('Ecuador' == data['locale_name']))).astype(dtype='float64')
    
    # Merge oil prices into data and interpolate missing values
    data = data.merge(oil_price, how='left', left_on='date', right_on='date')
    data.sort_values('id', inplace=True)
    data['dcoilwtico'] = data['dcoilwtico'].ffill()
    
    # Merge transactions into data
    data = data.merge(transactions, how='left', left_on=['date', 'store_nbr'], right_on=['date', 'store_nbr'])
    
    # Including day of week and pay day information
    data['date'] = pd.to_datetime(data['date'])
    data['day'] = data['date'].dt.day
    data['month'] = data['date'].dt.month
    data['year'] = data['date'].dt.year
    data['day_of_week'] = data['date'].dt.day_name()
    data['DoW'] = data['date'].dt.dayofweek
    data['Pay_Day'] = ((data['date'].dt.day == 1) | (data['date'].dt.day == 16)).astype(dtype='float64')
    data['Earthquake'] = (data['date'] == '2016-04-16').astype(dtype='float64')
    
    # Fill NaN values with proper substitute values
    data.fillna({'Holiday_Gen': False, 'Event_Gen': False, 'Work_Day_Gen': False, 'Holiday': 0.0, 'Event': 0.0,
                 'Work_Day': 0.0, 'Pay_Day': 0.0, 'Earthquake': 0.0, 'transactions': 0.0}, inplace=True)
    
    # One Hot Encoding of state related features
    encoder = OneHotEncoder()
    One_Hot_Coding = pd.DataFrame(encoder.fit_transform(data[['store_nbr', 'family', 'day_of_week']]).toarray(),
                                  columns=encoder.get_feature_names_out()) #.astype(dtype=np.int32) 'city', 'type', 'cluster',
    
    # Scaling numeric values
    if fit_scaler:
        scalers['scaler_dcoilwtico'].fit(oil_price[['dcoilwtico']])
        for col in ['onpromotion', 'day', 'month', 'year', 'DoW']:
            scalers[f'scaler_{col}'].fit(data[[col]])
    data[['dcoilwtico']] = scalers['scaler_dcoilwtico'].transform(data[['dcoilwtico']])
    for col in ['onpromotion', 'day', 'month', 'year', 'DoW']:
        data[[col]] = scalers[f'scaler_{col}'].transform(data[[col]])
    
    # Constructing feature vectors and labels vector
    features = data[['onpromotion', 'dcoilwtico', 'Holiday', 'Event', 'Work_Day', 'Earthquake', 'day', 'month', 'year']]
    features = features.merge(One_Hot_Coding, how='left', left_index=True, right_index=True)
    
    if 'sales' in data.columns: 
        labels = data[['sales']]
    else: labels = []
    return features, labels, scalers, data

trainvaltes_feat, trainvaltes_label, scalers, datat = process_data(train, stores, transactions, holidays, oil_price, True, scalers)
test_feat, test_label, _, data = process_data(test, stores, transactions, holidays, oil_price, False, scalers)

In [ ]:
## Spliting the train data into train, val and tes

skf5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2023)
gen_cv = skf5.split(trainvaltes_feat, np.array(datat['store_nbr']), np.array(datat['date']))
skf20 = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=2023)
for fold, (train_idx, val_idx) in enumerate(skf20.split(trainvaltes_feat, np.array(datat['store_nbr']),
                                                        np.array(datat['date']))):
    train_feat, train_label = trainvaltes_feat.iloc[train_idx], trainvaltes_label.iloc[train_idx]
    val_feat, val_label = trainvaltes_feat.iloc[val_idx], trainvaltes_label.iloc[val_idx]
    print(f"✅ Fold {fold}: Train size = {len(train_idx)}, Val size = {len(val_idx)}")
    break  # Use only the first fold for now

# Verify sizes
print(f"Total rows:   {len(trainvaltes_feat)}")
print(f"Dev train:    {len(train_feat)} ({len(train_feat)/len(trainvaltes_feat):.2%})")
print(f"Dev valid:    {len(val_feat)} ({len(val_feat)/len(trainvaltes_feat):.2%})")

# 3. Define model space for ML methods and NN architectures

In [ ]:
## Helping function to create parameter grids

def make_param(param_dict, model):
    for elem in param_dict.copy():
        if elem == 'n_components':
            param_dict['pca'+'__'+elem] = param_dict.pop(elem)
        else:
            param_dict[model+'__'+elem] = param_dict.pop(elem)
    return param_dict

In [ ]:
## Machine learning models and their hyperparameter search space

# Models
svr = SVR(kernel='rbf')
rfr = RandomForestRegressor(n_estimators=100)
kneigh = KNeighborsRegressor(n_neighbors=5)
gbr = GradientBoostingRegressor(n_estimators=200)
xgb = xgboost.XGBRegressor(device='cpu', eval_metric='mae', seed=0)
ada = AdaBoostRegressor(random_state=42)
hgbr = HistGradientBoostingRegressor(random_state=42)

# Model space
EstimatorStr = {1: 'svr', 2: 'rfr', 3: 'kneigh', 4: 'gbr', 5: 'xgb', 6: 'tbd'}
EstimatorMdl = {1: svr, 2: rfr, 3: kneigh, 4: gbr, 5: xgb}

# Tuned hyperparameter sets
param_single_svr = {'svr__C': 1, 'svr__degree': 3, 'svr__kernel': 'rbf'}
param_single_rfr = {'randomforestregressor__n_estimators': 100}
param_single_kneigh = {'kneighborsregressor__n_neighbors': 5}
param_single_gbr = {'gradientboostingregressor__n_estimators': 100, 'gradientboostingregressor__max_depth': 3}
param_single_xgb = {'xgbregressor__n_estimators': 100, 'xgbregressor__max_depth': 3}
param_single_ada = {}
param_single_hgbr = {}

# Hyperparameter search sets for machine learning methods
param_grid_svr = {'svr__C': np.linspace(1, 10, 5), 'svr__degree':  np.arange(2, 6, 1), 'svr__kernel': ['rbf']}
param_grid_rfr = {'randomforestregressor__n_estimators': np.arange(50, 250, 50)}
param_grid_kneigh = {'kneighborsregressor__n_neighbors': np.arange(2, 12, 2)}
param_grid_gbr = {'gradientboostingregressor__n_estimators': np.arange(50, 250, 50),
                  'gradientboostingregressor__max_depth': np.arange(2, 6, 1)}
param_grid_xgb = {'xgbregressor__n_estimators': np.arange(50, 250, 50), 'xgbregressor__max_depth': np.arange(2, 6, 1)}
param_grid_ada = {}
param_grid_hgbr = {}

In [ ]:
## Neural network model (DNN)

def build_network(hp):
    # define the sets of inputs
    inp = tf.keras.Input(shape=(len(train_feat.columns),), name='input')
    x = inp
    
    # apply FC layers and then a regression
    reg = None #tf.keras.regularizers.L2(hp.Float(name='l2', min_value=5e-4, max_value=5e-2, default=0.025))
    do = hp.Float(name='dropout', min_value=0.04, max_value=0.04, default=0.04)
    layers = hp.Int(name='layers', min_value=5, max_value=5, default=5)
    units = hp.Int(name='units', min_value=768, max_value=768, step = 128, default=768)
    act = 'relu'
    ki = 'he_normal'
    for i, units in enumerate([units for i in range(layers)]):
        x = tf.keras.layers.Dense(units, activation=act, kernel_regularizer=reg, kernel_initializer=ki,
                                  name=f'fc{i+1}')(x)
        x = tf.keras.layers.Dropout(do, name=f'do{i+1}')(x)
    out = tf.keras.layers.Dense(1, activation='relu', kernel_regularizer=reg, kernel_initializer=ki,
                                name='regression')(x)

    # define model
    model = tf.keras.Model(inputs=inp, outputs=out, name='DNN_network')
    
    # define optimizer/loss and compile model
    lr_tune = hp.Float(name='learning_rate', min_value=5e-5, max_value=5e-3, sampling='log', default=6e-4)
    optimizer = tf.keras.optimizers.Adam(lr_tune)
    model.compile(optimizer=optimizer, loss=tf.keras.losses.msle, metrics=['msle'])
    return model

with strategy.scope():
    model_nn = build_network(kt.HyperParameters())

model_nn.summary()

# 4. Training

Owing to the large amount of samples, the classical ML methods does not seem to be a good option. However for experimenting purposes I have included an ML part as well where one can experiment with several ML algorithm. For submission only NN network is activated.

In [ ]:
## Training configurations

TUNING = False
TRAINING = True
NN = True
ML = False # Only for experimenting purposes as benchmark

In [ ]:
## Hyperparameter search or single fitting for machine learning methods

for i_TypEstimator in [5]: # Choosing models to fit or tune {1: svc, 2: rfc, 3: kneigh, 4: gbc, 5: xgb} 1, 2, 3, 4, 5
    if ML: 
        if TUNING:
            param = globals()[f'param_grid_{EstimatorStr[i_TypEstimator]}']
        else:
            param = globals()[f'param_single_{EstimatorStr[i_TypEstimator]}']
            
        if ('pca__n_components' in param.keys() and param['pca__n_components']):
            param.pop('pca__n_components')
            pl = make_pipeline(EstimatorMdl[i_TypEstimator])
        elif 'pca__n_components' not in param.keys():
            pl = make_pipeline(EstimatorMdl[i_TypEstimator])
        else:
            pca = PCA(random_state=42)
            pl = make_pipeline(pca, EstimatorMdl[i_TypEstimator])
        pipeline = deepcopy(pl)

        if TUNING:
            grid = GridSearchCV(pipeline, param, scoring='neg_root_mean_squared_error', verbose=3, cv=gen_cv)
            grid.fit(trainvaltes_feat, np.array(trainvaltes_label).ravel())
            print(grid.best_params_)
            model_ml = grid.best_estimator_
        else:
            pipeline.set_params(**param)
            pipeline.fit(train_feat, np.array(train_label).ravel())
            model_ml = pipeline

        # Custom metric
        train_preds = np.clip(model_ml.predict(train_feat), a_min=0, a_max=None)
        val_preds = np.clip(model_ml.predict(val_feat), a_min=0, a_max=None)
        train_msle = np.sqrt(mean_squared_log_error(train_preds, np.array(train_label).ravel()))
        val_msle = np.sqrt(mean_squared_log_error(val_preds, np.array(val_label).ravel()))
        print(f'Train score: {train_msle}')
        print(f'Validation score: {val_msle}')
        globals()[f'model_{EstimatorStr[i_TypEstimator]}'] = model_ml

In [ ]:
## Training parameters for DNN network

epochs = 100

# Callback functions
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=5, verbose=1, monitor='val_msle')
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=10, verbose=1, monitor='val_msle', restore_best_weights=True)
lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-5 * 10**(epoch / 20)) # Find starting learning

In [ ]:
## Tuner configuration

i_TunerTyp = 3 # Choose desired tuner type: {1: 'grid', 2: 'random', 3: 'hyper'}
TunerStr = {1: 'grid', 2: 'random', 3: 'hyper'}

tuner_grid = kt.GridSearch(hypermodel=build_network, objective='val_msle', max_trials=10, overwrite=True,
                      directory="tuner", project_name="StoreSales", distribution_strategy = strategy)

tuner_random = kt.RandomSearch(hypermodel=build_network, objective='val_msle', max_trials=10, executions_per_trial=1,
                               overwrite=True, directory="tuner", project_name="StoreSales",
                               distribution_strategy = strategy)

tuner_hyper = kt.Hyperband(hypermodel=build_network, objective='val_msle', max_epochs=50, factor=3,
                           hyperband_iterations=1, directory="tuner", project_name="StoreSales",
                           distribution_strategy = strategy)

tuner = globals()[f'tuner_{TunerStr[i_TunerTyp]}']
tuner.search_space_summary()

In [ ]:
## Train model by Tuner or Fit

if TUNING and NN:
    tuner.search(train_feat, train_label, validation_data=(val_feat, val_label), batch_size=1024,
                 epochs=epochs, callbacks=[lr_scheduler, early_stopping_cb])
    best_models = tuner.get_best_models(num_models=2)
    model_nn = best_models[0]
    model_nn.summary()
    tuner.results_summary()

if TRAINING and NN:
    history = model_nn.fit(train_feat, train_label, validation_data=(val_feat, val_label), batch_size=1024,
                        epochs=epochs, callbacks=[lr_scheduler, early_stopping_cb])
    model_nn.evaluate(val_feat, val_label)

# 5. Evaluation

In [ ]:
## Plot learning curves

if TRAINING and NN:
    history_fil = {key: history.history[key] for key in ['msle', 'val_msle']}
    history_fil2 = {key: history.history[key] for key in ['loss', 'val_loss']}
    history_fil3 = {key: history.history[key] for key in ['learning_rate']}
                                                      
    pd.DataFrame(history_fil).plot()
    plt.ylabel("MSLE")
    plt.xlabel("epochs")
    plt.axis([3, len(history_fil['val_msle']), 0, history_fil['val_msle'][3]+0.1*history_fil['val_msle'][3]])
    pd.DataFrame(history_fil2).plot()
    plt.ylabel("Loss")
    plt.xlabel("epochs")
    plt.axis([3, len(history_fil2['val_loss']), 0, history_fil2['val_loss'][3]+0.1*history_fil2['val_loss'][3]])
    pd.DataFrame(history_fil3).plot()
    plt.ylabel("Learning rate")
    plt.xlabel("epochs")

In [ ]:
## Plot sales for selected store and product family

store = 44
family = 'AUTOMOTIVE' # SEAFOOD AUTOMOTIVE
store_train = f'store_nbr_{store}'
family_train = f'family_{family}'
datat_fil = datat[((datat['store_nbr']==store) & (datat['family']==family))]
train_feat_fil = train_feat[((train_feat[store_train]==True) & (train_feat[family_train]==True))]
train_pred_fil = pd.DataFrame(model_nn.predict(train_feat_fil), columns=['prediction'], index=train_feat_fil.index)
datat_fil = datat_fil.merge(train_pred_fil, how='left', left_index=True, right_index=True)
datat_fil.set_index(['date'], inplace=True)

start = 31
plt.figure()
datat_fil[['sales', 'prediction']][start:31+start].plot(title='Sales prediction of store {} for {} products'.format(str(store), family))
plt.figure()
datat_fil[['sales', 'prediction']][start:90+start].plot(title='Sales prediction of store {} for {} products'.format(str(store), family))
plt.figure()
datat_fil[['sales', 'prediction']][start:356+start].plot(title='Sales prediction of store {} for {} products'.format(str(store), family))

# 6. Submission

In [ ]:
## Make predictions and generate submission file

models=[]
if ML:
    models = models + ['svc', 'rfc', 'kneigh', 'gbc', 'xgb']
if NN:
    models = models + ['nn']

submission_all = pd.DataFrame(data[['id']])
for mod in models:
    model = globals()[f'model_{mod}']
    submission = pd.DataFrame(data[['id']])
    if mod == 'nn':
        predictions = np.stack([model(test_feat, training=True) for sample in range(200)])
        prediction = predictions.mean(axis=0)
    else:
        prediction = model.predict(test_feat)
    test_prediction = pd.DataFrame(prediction, columns=['sales'])
    submission = submission.merge(test_prediction, how='left', left_index=True, right_index=True)
    submission.to_csv(f'submission_{mod}.csv', index=False)
    submission_all[[mod]] = submission[['sales']]
if ML and NN:
    submission_all[['sales']] = pd.DataFrame((submission_all[['svc', 'rfc', 'gbc', 'xgb', 'nn']].sum(axis=1)/5))
    submission_all[['id', 'sales']].to_csv('submission_avg.csv', index=False)

# 7. Experimenting code

In [ ]:
# # Plot learning curves for definition of start leraning rate
# lrs = 1e-6 * (10 ** (np.arange(100) / 20)) # Define the learning rate array
# plt.figure(figsize=(10, 6)) # Set the figure size
# plt.grid(True) # Set the grid
# plt.semilogx(lrs, history.history["loss"]) # Plot the loss in log scale
# plt.tick_params('both', length=10, width=1, which='both') # Increase the tickmarks size
# plt.axis([1e-5, 1e-1, 0, 1]) # Set the plot boundaries

In [ ]:
#train_feat, val_feat, train_label, val_label = train_test_split(trainvaltes_feat, trainvaltes_label, test_size=0.005, random_state = 33)